<a href="https://colab.research.google.com/github/cn8972/Echo-Bot/blob/main/AI_Assisted_Cognitive_Delegation_Explorer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# AI-Assisted Cognitive Delegation Explorer
# Google Colab Version with WildChat-1M Dataset Included
# Dataset: allenai/WildChat-1M
# ============================================================

!pip install -q datasets gradio scikit-learn pandas numpy matplotlib

import re
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    mean_absolute_error,
    r2_score
)

warnings.filterwarnings("ignore")


# ============================================================
# 1. Research Framing
# ============================================================

RESEARCH_QUESTION = """
To what extent can conversational interaction patterns in large-scale human-AI chat logs
serve as observable indicators of AI-assisted cognitive delegation?
"""


# ============================================================
# 2. Load WildChat-1M Dataset
# ============================================================

# This is the exact dataset call requested.
# It may take time because WildChat-1M is a large dataset.
ds = load_dataset("allenai/WildChat-1M")

print(ds)


# ============================================================
# 3. Select a Manageable Sample for Colab
# ============================================================

SAMPLE_SIZE = 5000

# WildChat-1M normally includes a train split.
# This keeps the notebook manageable in Google Colab.
raw_df = ds["train"].select(range(SAMPLE_SIZE)).to_pandas()

print("Raw dataset shape:", raw_df.shape)
print("Dataset columns:", raw_df.columns.tolist())
raw_df.head()


# ============================================================
# 4. Define Cognitive Offloading Categories
# ============================================================

OFFLOADING_TYPES = {
    "Memory Offloading": [
        "summarize",
        "recap",
        "remember",
        "organize my notes",
        "take notes",
        "outline this",
        "extract key points",
        "condense",
        "bullet points",
        "tl;dr",
        "make a summary"
    ],
    "Reasoning Offloading": [
        "solve",
        "explain",
        "analyze",
        "reason through",
        "why is",
        "how does",
        "calculate",
        "derive",
        "interpret",
        "evaluate",
        "logic",
        "prove"
    ],
    "Writing Offloading": [
        "write",
        "draft",
        "compose",
        "rewrite",
        "enhance",
        "polish",
        "create talking points",
        "write an email",
        "write a letter",
        "write a paper",
        "generate text",
        "make this professional"
    ],
    "Decision Offloading": [
        "which should i choose",
        "what should i do",
        "recommend",
        "best option",
        "decide",
        "choose for me",
        "pros and cons",
        "should i",
        "which is better",
        "rank these"
    ],
    "Verification Offloading": [
        "check",
        "review",
        "is this correct",
        "verify",
        "validate",
        "debug",
        "find errors",
        "proofread",
        "fact check",
        "does this make sense",
        "critique"
    ],
    "Planning Offloading": [
        "create a plan",
        "build a plan",
        "schedule",
        "timeline",
        "roadmap",
        "study plan",
        "project plan",
        "lesson plan",
        "workflow",
        "steps",
        "organize a process"
    ]
}


# ============================================================
# 5. Text Extraction from WildChat Conversations
# ============================================================

def normalize_text(text):
    """
    Normalize text for rule-based feature extraction.
    """
    return re.sub(r"\s+", " ", str(text).lower().strip())


def extract_user_text_from_row(row):
    """
    Extracts user messages from WildChat-1M rows.

    WildChat-1M records may include conversation-style fields.
    This function checks common fields and extracts human/user messages.
    """

    possible_conversation_columns = [
        "conversation",
        "conversations",
        "messages",
        "turns"
    ]

    for col in possible_conversation_columns:
        if col in row.index and row[col] is not None:
            conv = row[col]

            if isinstance(conv, str):
                try:
                    conv = json.loads(conv)
                except Exception:
                    return str(conv)

            user_messages = []

            if isinstance(conv, list):
                for turn in conv:
                    if isinstance(turn, dict):
                        role = str(
                            turn.get("role") or
                            turn.get("from") or
                            turn.get("speaker") or
                            ""
                        ).lower()

                        content = (
                            turn.get("content") or
                            turn.get("value") or
                            turn.get("text") or
                            turn.get("message") or
                            ""
                        )

                        if "user" in role or "human" in role:
                            user_messages.append(str(content))

                if len(user_messages) > 0:
                    return " ".join(user_messages)

            return str(conv)

    fallback_text_columns = [
        "prompt",
        "instruction",
        "text",
        "user",
        "query",
        "input"
    ]

    for col in fallback_text_columns:
        if col in row.index and row[col] is not None:
            return str(row[col])

    return ""


def prepare_interaction_log_dataframe(raw_df):
    """
    Converts the raw WildChat dataset into an interaction-log dataframe.
    """
    records = []

    for idx, row in raw_df.iterrows():
        user_text = extract_user_text_from_row(row)

        records.append({
            "record_id": idx,
            "user_text": user_text,
            "text_length": len(str(user_text)),
            "word_count": len(str(user_text).split())
        })

    df = pd.DataFrame(records)

    df = df[df["user_text"].astype(str).str.len() > 0].reset_index(drop=True)

    return df


interaction_df = prepare_interaction_log_dataframe(raw_df)

print("Prepared interaction dataset shape:", interaction_df.shape)
interaction_df.head()


# ============================================================
# 6. Behavioral Proxy Feature Engineering
# ============================================================

def count_keyword_matches(text, keywords):
    """
    Counts how many keywords or phrases appear in a text.
    """
    text_norm = normalize_text(text)
    return sum(1 for keyword in keywords if keyword in text_norm)


def classify_offloading_type_rule(text):
    """
    Uses weak supervision to assign a cognitive offloading category.
    """
    text_norm = normalize_text(text)

    scores = {}

    for category, keywords in OFFLOADING_TYPES.items():
        scores[category] = count_keyword_matches(text_norm, keywords)

    best_category = max(scores, key=scores.get)
    best_score = scores[best_category]

    if best_score == 0:
        return "Low or Unclear Offloading"

    return best_category


def compute_behavioral_proxy_features(df):
    """
    Creates observable behavioral proxy variables from interaction logs.
    """
    df = df.copy()

    df["clean_text"] = df["user_text"].apply(normalize_text)

    df["question_mark_count"] = df["user_text"].astype(str).str.count(r"\?")

    df["imperative_count"] = df["clean_text"].apply(
        lambda x: sum(
            x.startswith(verb)
            for verb in [
                "write",
                "create",
                "make",
                "build",
                "give",
                "generate",
                "summarize",
                "explain",
                "solve",
                "check",
                "review",
                "develop",
                "draft",
                "recommend",
                "analyze",
                "compare"
            ]
        )
    )

    df["memory_proxy_count"] = df["clean_text"].apply(
        lambda x: count_keyword_matches(x, OFFLOADING_TYPES["Memory Offloading"])
    )

    df["reasoning_proxy_count"] = df["clean_text"].apply(
        lambda x: count_keyword_matches(x, OFFLOADING_TYPES["Reasoning Offloading"])
    )

    df["writing_proxy_count"] = df["clean_text"].apply(
        lambda x: count_keyword_matches(x, OFFLOADING_TYPES["Writing Offloading"])
    )

    df["decision_proxy_count"] = df["clean_text"].apply(
        lambda x: count_keyword_matches(x, OFFLOADING_TYPES["Decision Offloading"])
    )

    df["verification_proxy_count"] = df["clean_text"].apply(
        lambda x: count_keyword_matches(x, OFFLOADING_TYPES["Verification Offloading"])
    )

    df["planning_proxy_count"] = df["clean_text"].apply(
        lambda x: count_keyword_matches(x, OFFLOADING_TYPES["Planning Offloading"])
    )

    df["offloading_type"] = df["clean_text"].apply(classify_offloading_type_rule)

    proxy_columns = [
        "memory_proxy_count",
        "reasoning_proxy_count",
        "writing_proxy_count",
        "decision_proxy_count",
        "verification_proxy_count",
        "planning_proxy_count",
        "question_mark_count",
        "imperative_count"
    ]

    df["raw_delegation_score"] = df[proxy_columns].sum(axis=1)

    max_score = max(df["raw_delegation_score"].max(), 1)

    df["cognitive_delegation_index"] = (
        df["raw_delegation_score"] / max_score * 100
    ).round(2)

    df["delegation_intensity"] = pd.cut(
        df["cognitive_delegation_index"],
        bins=[-1, 20, 50, 100],
        labels=["Low", "Moderate", "High"]
    ).astype(str)

    return df


interaction_df = compute_behavioral_proxy_features(interaction_df)

print("Interaction dataset with proxy features:", interaction_df.shape)
interaction_df[
    [
        "record_id",
        "user_text",
        "offloading_type",
        "delegation_intensity",
        "cognitive_delegation_index"
    ]
].head()


# ============================================================
# 7. Feature Columns and Targets
# ============================================================

FEATURE_COLUMNS = [
    "user_text",
    "text_length",
    "word_count",
    "question_mark_count",
    "imperative_count",
    "memory_proxy_count",
    "reasoning_proxy_count",
    "writing_proxy_count",
    "decision_proxy_count",
    "verification_proxy_count",
    "planning_proxy_count"
]

NUMERIC_FEATURES = [
    "text_length",
    "word_count",
    "question_mark_count",
    "imperative_count",
    "memory_proxy_count",
    "reasoning_proxy_count",
    "writing_proxy_count",
    "decision_proxy_count",
    "verification_proxy_count",
    "planning_proxy_count"
]

TEXT_FEATURE = "user_text"

TARGET_OFFLOADING_TYPE = "offloading_type"
TARGET_DELEGATION_INTENSITY = "delegation_intensity"
TARGET_DELEGATION_INDEX = "cognitive_delegation_index"


# ============================================================
# 8. Train AI/ML Models
# ============================================================

def train_models(df):
    """
    Trains three models:
    1. Offloading type classifier
    2. Delegation intensity classifier
    3. Cognitive delegation index regressor
    """

    model_df = df.copy()

    X = model_df[FEATURE_COLUMNS]
    y_type = model_df[TARGET_OFFLOADING_TYPE]
    y_intensity = model_df[TARGET_DELEGATION_INTENSITY]
    y_index = model_df[TARGET_DELEGATION_INDEX]

    stratify_target = y_type

    X_train, X_test, y_type_train, y_type_test = train_test_split(
        X,
        y_type,
        test_size=0.2,
        random_state=42,
        stratify=stratify_target
    )

    _, _, y_intensity_train, y_intensity_test = train_test_split(
        X,
        y_intensity,
        test_size=0.2,
        random_state=42,
        stratify=stratify_target
    )

    _, _, y_index_train, y_index_test = train_test_split(
        X,
        y_index,
        test_size=0.2,
        random_state=42,
        stratify=stratify_target
    )

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "text",
                TfidfVectorizer(
                    max_features=3000,
                    ngram_range=(1, 2),
                    stop_words="english"
                ),
                TEXT_FEATURE
            ),
            (
                "numeric",
                StandardScaler(),
                NUMERIC_FEATURES
            )
        ]
    )

    offloading_type_model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", LogisticRegression(
                max_iter=2000,
                class_weight="balanced"
            ))
        ]
    )

    intensity_model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", RandomForestClassifier(
                n_estimators=300,
                random_state=42,
                class_weight="balanced",
                min_samples_leaf=2,
                n_jobs=-1
            ))
        ]
    )

    index_model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", RandomForestRegressor(
                n_estimators=300,
                random_state=42,
                min_samples_leaf=2,
                n_jobs=-1
            ))
        ]
    )

    offloading_type_model.fit(X_train, y_type_train)
    intensity_model.fit(X_train, y_intensity_train)
    index_model.fit(X_train, y_index_train)

    type_predictions = offloading_type_model.predict(X_test)
    intensity_predictions = intensity_model.predict(X_test)
    index_predictions = index_model.predict(X_test)

    metrics = {
        "offloading_type_accuracy": accuracy_score(y_type_test, type_predictions),
        "delegation_intensity_accuracy": accuracy_score(y_intensity_test, intensity_predictions),
        "delegation_index_mae": mean_absolute_error(y_index_test, index_predictions),
        "delegation_index_r2": r2_score(y_index_test, index_predictions),
        "offloading_type_report": classification_report(
            y_type_test,
            type_predictions,
            zero_division=0
        ),
        "delegation_intensity_report": classification_report(
            y_intensity_test,
            intensity_predictions,
            zero_division=0
        )
    }

    trained_objects = {
        "offloading_type_model": offloading_type_model,
        "intensity_model": intensity_model,
        "index_model": index_model,
        "metrics": metrics,
        "X_train": X_train,
        "X_test": X_test,
        "y_type_test": y_type_test,
        "y_intensity_test": y_intensity_test,
        "y_index_test": y_index_test
    }

    return trained_objects


models = train_models(interaction_df)

print("Model training complete.")
print("Offloading Type Accuracy:", models["metrics"]["offloading_type_accuracy"])
print("Delegation Intensity Accuracy:", models["metrics"]["delegation_intensity_accuracy"])
print("Delegation Index MAE:", models["metrics"]["delegation_index_mae"])
print("Delegation Index R2:", models["metrics"]["delegation_index_r2"])


# ============================================================
# 9. Store Application State
# ============================================================

APP_STATE = {
    "df": interaction_df,
    "models": models
}


# ============================================================
# 10. Single Prompt Prediction
# ============================================================

def engineer_single_prompt(prompt):
    temp_df = pd.DataFrame({
        "record_id": [0],
        "user_text": [prompt],
        "text_length": [len(str(prompt))],
        "word_count": [len(str(prompt).split())]
    })

    temp_df = compute_behavioral_proxy_features(temp_df)

    return temp_df


def predict_prompt(prompt):
    if prompt is None or len(str(prompt).strip()) == 0:
        return "Please enter a prompt."

    temp_df = engineer_single_prompt(prompt)
    X_input = temp_df[FEATURE_COLUMNS]

    predicted_type = APP_STATE["models"]["offloading_type_model"].predict(X_input)[0]
    predicted_intensity = APP_STATE["models"]["intensity_model"].predict(X_input)[0]
    predicted_index = APP_STATE["models"]["index_model"].predict(X_input)[0]

    rule_type = temp_df["offloading_type"].iloc[0]
    rule_index = temp_df["cognitive_delegation_index"].iloc[0]

    return f"""
Predicted Offloading Type: {predicted_type}

Predicted Delegation Intensity: {predicted_intensity}

Predicted Cognitive Delegation Index: {predicted_index:.2f} / 100

Rule-Based Offloading Type: {rule_type}

Rule-Based Delegation Index: {rule_index:.2f} / 100

Interpretation:
This prompt contains observable linguistic and behavioral signals associated with {predicted_type.lower()}.
The delegation index estimates how strongly the prompt appears to delegate cognitive work to the AI system.
"""


# ============================================================
# 11. Dashboard Functions
# ============================================================

def summary_table():
    df = APP_STATE["df"]

    summary = df.groupby("offloading_type").agg(
        records=("record_id", "count"),
        mean_delegation_index=("cognitive_delegation_index", "mean"),
        median_delegation_index=("cognitive_delegation_index", "median"),
        mean_word_count=("word_count", "mean"),
        mean_question_marks=("question_mark_count", "mean"),
        mean_imperatives=("imperative_count", "mean")
    ).reset_index()

    numeric_cols = [
        "mean_delegation_index",
        "median_delegation_index",
        "mean_word_count",
        "mean_question_marks",
        "mean_imperatives"
    ]

    for col in numeric_cols:
        summary[col] = summary[col].round(2)

    return summary.sort_values("records", ascending=False)


def sample_records_by_type(offloading_type):
    df = APP_STATE["df"]

    if offloading_type != "All":
        df = df[df["offloading_type"] == offloading_type]

    cols = [
        "record_id",
        "user_text",
        "offloading_type",
        "delegation_intensity",
        "cognitive_delegation_index",
        "word_count"
    ]

    return df[cols].head(50)


def model_performance_text():
    metrics = APP_STATE["models"]["metrics"]

    return f"""
Research Question:
{RESEARCH_QUESTION}

Records Analyzed: {len(APP_STATE["df"]):,}

Model Performance Summary:

Offloading Type Accuracy: {metrics["offloading_type_accuracy"]:.3f}

Delegation Intensity Accuracy: {metrics["delegation_intensity_accuracy"]:.3f}

Delegation Index Mean Absolute Error: {metrics["delegation_index_mae"]:.3f}

Delegation Index R²: {metrics["delegation_index_r2"]:.3f}

Offloading Type Classification Report:

{metrics["offloading_type_report"]}

Delegation Intensity Classification Report:

{metrics["delegation_intensity_report"]}

Methodological Note:
These labels are weak labels generated from observable linguistic proxies. They should be interpreted as behavioral indicators of cognitive delegation, not direct psychological measures of internal cognition.
"""


def plot_offloading_distribution():
    df = APP_STATE["df"]
    counts = df["offloading_type"].value_counts()

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(counts.index[::-1], counts.values[::-1])
    ax.set_title("Offloading Type Distribution")
    ax.set_xlabel("Record Count")
    ax.set_ylabel("Offloading Type")
    plt.tight_layout()

    return fig


def plot_intensity_distribution():
    df = APP_STATE["df"]
    counts = df["delegation_intensity"].value_counts().reindex(
        ["Low", "Moderate", "High"]
    ).fillna(0)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(counts.index, counts.values)
    ax.set_title("Delegation Intensity Distribution")
    ax.set_xlabel("Delegation Intensity")
    ax.set_ylabel("Record Count")
    plt.tight_layout()

    return fig


def plot_index_histogram():
    df = APP_STATE["df"]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(df["cognitive_delegation_index"], bins=20)
    ax.set_title("Cognitive Delegation Index Distribution")
    ax.set_xlabel("Cognitive Delegation Index")
    ax.set_ylabel("Record Count")
    plt.tight_layout()

    return fig


def refresh_dashboard(offloading_type):
    return (
        summary_table(),
        sample_records_by_type(offloading_type),
        plot_offloading_distribution(),
        plot_intensity_distribution(),
        plot_index_histogram()
    )


# ============================================================
# 12. Dataset Chat
# ============================================================

def answer_dataset_question(message, history):
    df = APP_STATE["df"]
    text = normalize_text(message)

    if "research question" in text:
        return RESEARCH_QUESTION

    if "how many" in text or "count" in text or "records" in text:
        return f"The analyzed dataset contains {len(df):,} interaction records."

    if "offloading type" in text or "types" in text:
        counts = df["offloading_type"].value_counts().reset_index()
        counts.columns = ["offloading_type", "records"]
        return counts.to_markdown(index=False)

    if "intensity" in text:
        counts = df["delegation_intensity"].value_counts().reset_index()
        counts.columns = ["delegation_intensity", "records"]
        return counts.to_markdown(index=False)

    if "average" in text or "mean" in text:
        return f"The mean Cognitive Delegation Index is {df['cognitive_delegation_index'].mean():.2f}."

    if "median" in text:
        return f"The median Cognitive Delegation Index is {df['cognitive_delegation_index'].median():.2f}."

    if "top" in text or "highest" in text:
        top_df = df.sort_values(
            "cognitive_delegation_index",
            ascending=False
        ).head(10)

        return top_df[
            [
                "record_id",
                "offloading_type",
                "delegation_intensity",
                "cognitive_delegation_index",
                "user_text"
            ]
        ].to_markdown(index=False)

    if "help" in text:
        return """
You can ask:
- What is the research question?
- How many records are in the dataset?
- What are the offloading types?
- Show intensity distribution.
- What is the average cognitive delegation index?
- Show the top records by cognitive delegation index.
"""

    return """
I could not fully interpret that question. Try asking:
- What are the offloading types?
- What is the average cognitive delegation index?
- Show the top records by cognitive delegation index.
- What is the research question?
"""


def chat_submit(message, history):
    if history is None:
        history = []

    response = answer_dataset_question(message, history)

    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": response}
    ]

    return history, ""


# ============================================================
# 13. Gradio Application
# ============================================================

offloading_choices = [
    "All",
    "Memory Offloading",
    "Reasoning Offloading",
    "Writing Offloading",
    "Decision Offloading",
    "Verification Offloading",
    "Planning Offloading",
    "Low or Unclear Offloading"
]

with gr.Blocks(title="AI-Assisted Cognitive Delegation Explorer") as demo:

    gr.Markdown("# AI-Assisted Cognitive Delegation Explorer")

    gr.Markdown("""
This Google Colab application uses the `allenai/WildChat-1M` dataset to examine observable behavioral proxies of cognitive offloading in human-AI interaction logs.

**Research Question:**
To what extent can conversational interaction patterns in large-scale human-AI chat logs serve as observable indicators of AI-assisted cognitive delegation?

The application classifies user prompts into cognitive offloading categories, estimates delegation intensity, and provides an interactive dashboard for analysis.
""")

    with gr.Tab("Dataset Overview"):
        overview_textbox = gr.Textbox(
            label="Model and Dataset Summary",
            value=model_performance_text(),
            lines=30
        )

        preview_df = gr.Dataframe(
            label="Prepared Dataset Preview",
            value=APP_STATE["df"][
                [
                    "record_id",
                    "user_text",
                    "offloading_type",
                    "delegation_intensity",
                    "cognitive_delegation_index"
                ]
            ].head(100)
        )

    with gr.Tab("Dashboard"):
        selected_type = gr.Dropdown(
            choices=offloading_choices,
            value="All",
            label="Filter by Offloading Type"
        )

        refresh_button = gr.Button("Refresh Dashboard")

        summary_output = gr.Dataframe(label="Summary by Offloading Type")
        sample_output = gr.Dataframe(label="Sample Records")
        type_plot = gr.Plot(label="Offloading Type Distribution")
        intensity_plot = gr.Plot(label="Delegation Intensity Distribution")
        index_plot = gr.Plot(label="Cognitive Delegation Index Distribution")

        refresh_button.click(
            fn=refresh_dashboard,
            inputs=[selected_type],
            outputs=[
                summary_output,
                sample_output,
                type_plot,
                intensity_plot,
                index_plot
            ]
        )

    with gr.Tab("Prompt Prediction"):
        prompt_input = gr.Textbox(
            label="Enter a Human-AI Prompt",
            lines=8,
            placeholder="Example: Write a professional email explaining why this project timeline needs to be revised."
        )

        predict_button = gr.Button("Predict Cognitive Delegation Pattern")

        prediction_output = gr.Textbox(
            label="Prediction Output",
            lines=14
        )

        predict_button.click(
            fn=predict_prompt,
            inputs=[prompt_input],
            outputs=[prediction_output]
        )

    with gr.Tab("Dataset Chat"):
        gr.Markdown("""
Ask questions about the analyzed WildChat interaction-log dataset.

Examples:
- What are the offloading types?
- What is the average cognitive delegation index?
- Show the top records by cognitive delegation index.
- What is the research question?
""")

        chatbot = gr.Chatbot(
            label="Dataset Analyst",
            type="messages",
            height=500
        )

        chat_input = gr.Textbox(
            label="Ask a question about the dataset"
        )

        send_button = gr.Button("Send")
        clear_button = gr.Button("Clear Chat")

        send_button.click(
            fn=chat_submit,
            inputs=[chat_input, chatbot],
            outputs=[chatbot, chat_input]
        )

        chat_input.submit(
            fn=chat_submit,
            inputs=[chat_input, chatbot],
            outputs=[chatbot, chat_input]
        )

        clear_button.click(
            fn=lambda: [],
            inputs=[],
            outputs=[chatbot]
        )


# ============================================================
# 14. Launch Application
# ============================================================

demo.launch(share=True)

In [3]:
summary_df = summary_table()
display(summary_df)

,offloading_type,records,mean_delegation_index,median_delegation_index,mean_word_count,mean_question_marks,mean_imperatives
1,Low or Unclear Offloading,2749,0.89,0.00,371.84,0.72,0.0
4,Reasoning Offloading,1106,8.23,4.94,1531.27,3.14,0.0
6,Writing Offloading,589,4.52,2.47,815.17,1.90,0.0
2,Memory Offloading,217,7.27,4.94,1113.51,3.22,0.0
5,Verification Offloading,195,5.05,2.47,1017.39,1.86,0.0
0,Decision Offloading,89,7.85,4.94,997.72,3.58,0.0
3,Planning Offloading,55,3.50,2.47,695.38,1.16,0.0
